In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import polars as pl


In [14]:
shell = r'C:\Users\187697\Dropbox\sa_fires\proj_bureaucrats_farms'
dbox = r'/Users/anzony.quisperojas/Library/CloudStorage/Dropbox/sa_fires/proj_bureaucrats_farms'
main = dbox
int_path = f'{main}/data_output/intermediate'
figures = f"{main}/tex/paper/figures"


In [ ]:
df = pd.read_stata(f"{int_path}/combined_dt_pop.dta")
dt = pl.from_pandas(df)

# merge is_rural, keep _merge==3, keep is_rural==1
ghs = pl.from_pandas(__import__("pandas").read_stata(
    f"{int_path}/ghs_grid_classification_2000.dta")).select(
    ["unique_small_grid_id", "is_rural"])

dt  = dt.with_columns(pl.col("unique_small_grid_id").cast(pl.Int64))
ghs = ghs.with_columns(pl.col("unique_small_grid_id").cast(pl.Int64))
dt = dt.join(ghs, on="unique_small_grid_id", how="inner")   # inner = _merge==3
dt = dt.filter(pl.col("is_rural") == 1)

# merge grids_with_more_1_ac, drop dpl_ac==1
multi = pl.from_pandas(__import__("pandas").read_stata(
    f"{int_path}/grids_with_more_1_ac.dta"))
multi = multi.with_columns(pl.col("unique_small_grid_id").cast(pl.Int64))
dt = dt.join(multi, on="unique_small_grid_id", how="left")
dt = dt.with_columns(pl.col("dpl_ac").cast(pl.Int64))
dt = dt.filter(pl.col("dpl_ac").is_null())   # conserva NA y 0, quita 1
# keep if year<2022 | (year==2022 & month<=8)
dt = dt.filter(
    (pl.col("year") < 2022) | ((pl.col("year") == 2022) & (pl.col("month") <= 8))
)

# ── 3. Collapse: número de observaciones por período ────────────────────
counts = (
    dt.group_by("relative_monthyear")
      .agg(pl.len().alias("n_obs"))
      .sort("relative_monthyear")
)
print(counts)


In [ ]:
# Sort by rtime for consistency
df = counts.to_pandas()
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.linewidth": 0.8,
    "xtick.direction": "out",
    "ytick.direction": "out",
})
# Create histogram manually using bar plot (since we already have aggregated counts)
plt.figure(figsize=(8,5))
# plt.bar(df["rtime"], df["nobs"]/1000, color="steelblue", edgecolor=None)

# Define bin edges with width = 5
edge = max(abs(df["relative_monthyear"].min()), abs( df["relative_monthyear"].max()))
bins = np.arange(-1*edge, edge + 5, 5)

# Weighted histogram — weight each rtime by its nobs
plt.hist(df["relative_monthyear"], bins=bins, weights=df["n_obs"]/1000,
         color="gray", edgecolor="black", alpha=0.8)

# Labels and style
plt.xlabel("Relative Time periods from Treatment")
plt.ylabel("Density of Observations per period")
# plt.title("Distribution of rtime (in thousands of observations)")
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
# Save or show the figure
plt.savefig(f"{figures}/downup_evtime_hist.png", dpi=300)
plt.show()